# The Green Divide: 20 Years of EU Renewable Energy
### Data analysis notebook · DataBites · Josep Ferrer

---

This notebook documents the full data pipeline behind the story
**"Europe's renewable energy divide is getting wider, not narrower"**,
published on [reads.databites.tech](https://reads.databites.tech).

**The story in one sentence:** Despite two decades of EU climate policy and
three successive Renewable Energy Directives, the gap between Europe's
greenest and least-green member states has widened from 38 percentage
points in 2004 to nearly 50 in 2024.

**Data source:** Eurostat `nrg_ind_ren` — Share of energy from renewable
sources (% of gross final energy consumption), 2004 to 2024.
Free, public, updated annually. Last updated March 2026.

**Output:** Four CSVs exported to `outputs/`, each mapped to a specific
Datawrapper visualisation.

---

### Pipeline overview

```
Eurostat API  ->  raw JSON  ->  flat DataFrame  ->  EU-27 filter  ->  analysis  ->  CSVs  ->  Datawrapper
```

| Step | What happens |
|------|-------------|
| 0 | Imports, paths, reference dictionaries |
| 1 | Fetch raw data from Eurostat (with local cache) |
| 2 | Parse JSON-STAT format into a flat DataFrame |
| 3 | Filter to EU-27, clean types, add ISO3 codes |
| 4 | Explore the data and find the story |
| 5 | Export four chart-ready CSVs |

## 0. Setup

Three things to initialise before any data work:

**1. Libraries.** Standard stack: `requests` for the API call, `pandas`
for data manipulation, `itertools` and `json` for parsing.

**2. Paths.** The notebook sits at `stories/the_green_divide/`.
Raw data goes into `data/` (committed to git for reproducibility).
Chart-ready exports go into `outputs/` (also committed).

**3. Reference dictionaries.** Two lookup tables used throughout:

- `EU27`: maps ISO 3166-1 alpha-2 codes (e.g. `SE`) to country names.
  Used to filter Eurostat's 40-country response down to EU-27 members only.
- `ISO2_TO_3`: maps alpha-2 to alpha-3 codes (e.g. `SE` to `SWE`).
  Datawrapper's choropleth maps require ISO3, not ISO2.

In [ ]:
import requests
import pandas as pd
import itertools
import json
from pathlib import Path

OUTPUT_DIR = Path("outputs")
DATA_DIR   = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# Datawrapper choropleth maps require ISO 3166-1 alpha-3 codes.
# Eurostat uses alpha-2. This dict handles the conversion.
ISO2_TO_3 = {
    "AT":"AUT","BE":"BEL","BG":"BGR","HR":"HRV","CY":"CYP",
    "CZ":"CZE","DK":"DNK","EE":"EST","FI":"FIN","FR":"FRA",
    "DE":"DEU","GR":"GRC","HU":"HUN","IE":"IRL","IT":"ITA",
    "LV":"LVA","LT":"LTU","LU":"LUX","MT":"MLT","NL":"NLD",
    "PL":"POL","PT":"PRT","RO":"ROU","SK":"SVK","SI":"SVN",
    "ES":"ESP","SE":"SWE"
}

# EU-27 member states (post-Brexit composition).
# Eurostat's nrg_ind_ren response includes 40 geo entities:
# EU27 aggregate, euro area aggregates, candidate countries, etc.
# We use this dict to filter down to the 27 member states only.
EU27 = {
    "AT":"Austria",    "BE":"Belgium",     "BG":"Bulgaria",
    "HR":"Croatia",    "CY":"Cyprus",      "CZ":"Czechia",
    "DK":"Denmark",    "EE":"Estonia",     "FI":"Finland",
    "FR":"France",     "DE":"Germany",     "GR":"Greece",
    "HU":"Hungary",    "IE":"Ireland",     "IT":"Italy",
    "LV":"Latvia",     "LT":"Lithuania",   "LU":"Luxembourg",
    "MT":"Malta",      "NL":"Netherlands", "PL":"Poland",
    "PT":"Portugal",   "RO":"Romania",     "SK":"Slovakia",
    "SI":"Slovenia",   "ES":"Spain",       "SE":"Sweden"
}

print(f"EU-27 countries loaded: {len(EU27)}")
print(f"Data dir:   {DATA_DIR.resolve()}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")

## 1. Fetch data from the Eurostat API

Eurostat exposes all its datasets via a REST API that returns JSON-STAT
format, a compact representation designed for multi-dimensional statistical
data. The endpoint pattern is:

```
https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/{dataset_id}
```

We request dataset `nrg_ind_ren` with three filters applied:

- `nrg_bal=REN`: overall renewable share only. The dataset also contains
  electricity-only, transport-only, and heating-only breakdowns.
- `unit=PC`: percentage of gross final energy consumption.
- `freq=A`: annual data. Monthly is available for some indicators.

**Caching strategy.** The raw JSON response is saved to `data/nrg_ind_ren_raw.json`
on the first run. Subsequent runs load from disk. This makes the notebook
fully reproducible without needing an internet connection, and preserves
the exact data version used for this analysis.

In [ ]:
def fetch_eurostat(dataset_id, params=None):
    """
    Fetch a dataset from the Eurostat JSON-STAT API.

    Parameters
    ----------
    dataset_id : str
        Eurostat dataset code, e.g. 'nrg_ind_ren'
    params : dict, optional
        Additional filter parameters to narrow the response

    Returns
    -------
    dict
        Raw JSON-STAT response from the Eurostat API
    """
    base = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
    url  = f"{base}/{dataset_id}"
    defaults = {"format": "JSON", "lang": "en"}
    if params:
        defaults.update(params)
    r = requests.get(url, params=defaults, timeout=30)
    r.raise_for_status()
    return r.json()


def eurostat_to_df(data):
    """
    Parse Eurostat JSON-STAT format into a flat pandas DataFrame.

    JSON-STAT stores values as a flat dict keyed by a single integer index
    that encodes a position in an N-dimensional array. Each dimension has
    an ordered list of category codes. To reconstruct the full table we:

    1. Enumerate all combinations of dimension indices (itertools.product)
    2. Convert each combination to a flat integer index
    3. Look up the value at that index
    4. Map dimension indices back to their category codes

    Missing values (structural zeros, not-collected cells) are absent from
    the 'value' dict. They are silently skipped and later dropped as NaN.

    Parameters
    ----------
    data : dict
        Raw JSON-STAT response dict from fetch_eurostat()

    Returns
    -------
    pd.DataFrame
        One row per non-missing observation, one column per dimension
        plus a 'value' column.
    """
    dims   = data["dimension"]
    size   = data["size"]
    ids    = data["id"]
    values = data["value"]
    codes  = [list(dims[d]["category"]["index"].keys()) for d in ids]

    rows = []
    for combo in itertools.product(*[range(s) for s in size]):
        # Convert N-dimensional index to flat integer position
        flat_idx, mult = 0, 1
        for i in reversed(range(len(size))):
            flat_idx += combo[i] * mult
            mult *= size[i]
        # Only add rows where a value exists (skip structural missing)
        if str(flat_idx) in values:
            row = {ids[i]: codes[i][combo[i]] for i in range(len(ids))}
            row["value"] = values[str(flat_idx)]
            rows.append(row)

    return pd.DataFrame(rows)


print("Functions defined.")

In [ ]:
RAW_FILE = DATA_DIR / "nrg_ind_ren_raw.json"

if RAW_FILE.exists():
    # Load from local cache - no API call needed
    print("Loading from cache...")
    with open(RAW_FILE) as f:
        raw = json.load(f)
else:
    # First run: fetch from Eurostat and cache locally
    print("Fetching from Eurostat API...")
    raw = fetch_eurostat("nrg_ind_ren", {
        "nrg_bal": "REN",   # Renewable energy - overall share
        "unit":    "PC",    # Percentage of gross final energy consumption
        "freq":    "A"      # Annual frequency
    })
    with open(RAW_FILE, "w") as f:
        json.dump(raw, f)
    print(f"Saved to {RAW_FILE}")

# Inspect the response structure
print(f"Dimensions:  {raw['id']}")
print(f"Size:        {raw['size']}")
print(f"Data points: {sum(1 for v in raw['value'].values() if v is not None)}")

## 2. Parse into a flat DataFrame

The JSON-STAT response has 5 dimensions: `freq`, `nrg_bal`, `unit`, `geo`,
and `time`. Since we filtered to a single value for the first three
(annual, REN, percentage), the only dimensions that vary are `geo`
(40 entities) and `time` (21 years: 2004 to 2024), giving a theoretical
maximum of 840 data points. The actual count is 777 because some
country-year combinations have no data.

After parsing, `eurostat_to_df()` returns one row per observation.
We verify the year range and inspect a sample to confirm the structure
before any filtering.

In [ ]:
df = eurostat_to_df(raw)

print(f"Shape:      {df.shape}")
print(f"Columns:    {df.columns.tolist()}")
print(f"Year range: {df['time'].min()} to {df['time'].max()}")
print(f"Geo codes:  {sorted(df['geo'].unique())[:10]} ...")
print()
print("Sample rows:")
df.head(8)

## 3. Filter to EU-27 and clean

Three cleaning steps applied in sequence:

**1. Filter geo.** Keep only the 27 EU member states. The raw response
includes EU and euro area aggregates (e.g. `EU27_2020`, `EA20`),
candidate countries (Turkey, Iceland, North Macedonia), and post-Brexit UK.
We exclude all of these for the member-state analysis, but retain
`EU27_2020` separately for the EU average reference line.

**2. Type conversion.** `time` to integer year, `value` to float.
The API returns both as strings. `errors="coerce"` turns any non-numeric
values (e.g. flagged estimates marked with `":"`) into NaN, which are
then dropped.

**3. Add ISO3 codes.** Datawrapper's choropleth requires alpha-3 country
codes. We map from the ISO2 codes in the Eurostat response using the
`ISO2_TO_3` dict defined in setup.

In [ ]:
# Filter to EU-27 member states only
df_eu = df[df["geo"].isin(EU27.keys())].copy()

# Add derived columns
df_eu["country"] = df_eu["geo"].map(EU27)
df_eu["iso3"]    = df_eu["geo"].map(ISO2_TO_3)
df_eu["year"]    = df_eu["time"].astype(int)
df_eu["pct"]     = pd.to_numeric(df_eu["value"], errors="coerce")

# Keep only the columns we need and drop any remaining NaNs
df_eu = df_eu[["geo", "iso3", "country", "year", "pct"]].dropna()

# Separately extract EU27 aggregate (used for the average reference line)
df_eu27 = df[df["geo"] == "EU27_2020"].copy()
df_eu27["year"] = df_eu27["time"].astype(int)
df_eu27["pct"]  = pd.to_numeric(df_eu27["value"], errors="coerce")

print(f"EU-27 member states with data: {df_eu['geo'].nunique()} (expected 27)")
print(f"Year range:                    {df_eu['year'].min()} to {df_eu['year'].max()}")
print(f"Total observations:            {len(df_eu)}")
print(f"EU27 aggregate rows:           {len(df_eu27.dropna(subset=['pct']))}")
print()
df_eu.sort_values(["country","year"]).head(10)

## 4. Explore: finding the story

Three analytical questions guide the exploration:

**Question 1:** Where does Europe stand in 2024? Which countries have already
met the 2030 target of 42.5%? How large is the gap between leader and laggard?

**Question 2:** Who moved the most over 20 years? Which countries accelerated
and which stagnated? This reveals whether the divide is structural or cyclical.

**Question 3:** Is the gap widening or narrowing? The headline claim of the
story. Confirmed if `gap_pp` grows from 2004 to 2024.

In [ ]:
# Question 1: 2024 snapshot
latest_year = df_eu["year"].max()
df_latest = df_eu[df_eu["year"] == latest_year].sort_values("pct", ascending=False)

top    = df_latest.iloc[0]
bottom = df_latest.iloc[-1]
eu_avg = df_eu27[df_eu27["year"] == latest_year]["pct"].values[0]

print(f"=== {latest_year} snapshot ===")
print(f"Leader:           {top['country']:15} {top['pct']:.1f}%")
print(f"Laggard:          {bottom['country']:15} {bottom['pct']:.1f}%")
print(f"Gap:              {top['pct'] - bottom['pct']:.1f} percentage points")
print(f"EU average:       {eu_avg:.1f}%")
print(f"2030 target:      42.5%")
print()

on_track  = df_latest[df_latest["pct"] >= 42.5]
off_track = df_latest[df_latest["pct"] <  42.5]
print(f"Already at or above 42.5% target: {len(on_track)} countries")
for _, row in on_track.iterrows():
    print(f"  {row['country']:15} {row['pct']:.1f}%")
print(f"\nBelow target with 6 years remaining: {len(off_track)} countries")

In [ ]:
# Question 2: Biggest movers 2004 to 2024
df_2004 = df_eu[df_eu["year"] == 2004].set_index("geo")["pct"]
df_now  = df_eu[df_eu["year"] == latest_year].set_index("geo")["pct"]

growth = (df_now - df_2004).dropna().sort_values(ascending=False).reset_index()
growth.columns = ["geo", "pp_change"]
growth["country"]  = growth["geo"].map(EU27)
growth["pct_2004"] = growth["geo"].map(df_2004).round(2)
growth["pct_now"]  = growth["geo"].map(df_now).round(2)

print(f"=== Change 2004 to {latest_year} (percentage points) ===")
print(f"{'Country':<15} {'2004':>8} {'2024':>8} {'Change':>8}")
print("-" * 45)
for _, row in growth.iterrows():
    print(f"{row['country']:<15} {row['pct_2004']:>8.1f} {row['pct_now']:>8.1f} {row['pp_change']:>+8.1f}")

In [ ]:
# Question 3: Is the gap widening?
annual = df_eu.groupby("year")["pct"]
gap_check = pd.DataFrame({
    "leader":  annual.max().round(2),
    "laggard": annual.min().round(2),
    "average": annual.mean().round(2),
    "gap_pp":  (annual.max() - annual.min()).round(2)
})

print("=== Gap between leader and laggard, by year ===")
print(f"{'Year':<6} {'Leader':>8} {'Laggard':>9} {'Gap (pp)':>10}")
print("-" * 38)
for year, row in gap_check.iterrows():
    print(f"{year:<6} {row['leader']:>8.1f} {row['laggard']:>9.1f} {row['gap_pp']:>10.1f}")

print(f"\n2004 gap: {gap_check['gap_pp'].iloc[0]:.1f} pp")
print(f"2024 gap: {gap_check['gap_pp'].iloc[-1]:.1f} pp")
print(f"Peak gap: {gap_check['gap_pp'].max():.1f} pp ({gap_check['gap_pp'].idxmax()})")

## 5. Export CSVs for Datawrapper

Five output files, each designed for a specific Datawrapper chart type.

| File | Chart type | What it shows |
|------|------------|---------------|
| `map_renewables_2024.csv` | Choropleth map | 2024 snapshot, all EU-27 |
| `timeseries_all_countries.csv` | Reference / all 27 lines | Full 20-year series |
| `timeseries_divide.csv` | Line chart, 6 countries | Leaders vs laggards, 2004 to 2024 |
| `gap_over_time.csv` | Area/line chart | Widening gap, min/max/avg by year |
| `timeseries_all_highlighted.csv` | Line chart, all 27 | All countries for grey-out effect |

**Format notes for Datawrapper:**

- Choropleth maps require ISO3 country codes in the key column.
  The column header must match Datawrapper's internal field name `ISO_3_SOV`.
- Line charts expect wide format: rows = years, columns = series names.
- All values are percentages (0 to 100), not decimals (0 to 1).
  Do not set the Datawrapper column type to "Percentage" as it would
  multiply values by 100.

In [ ]:
# CSV 1: Choropleth map
# One row per country. Key column = ISO3 (required by Datawrapper).
# Sorted alphabetically by ISO3 for readability.
map_df = df_latest[["iso3","country","pct"]].copy()
map_df.columns = ["ISO_3_SOV", "country", "renewable_pct_2024"]
map_df = map_df.sort_values("ISO_3_SOV")
map_df.to_csv(OUTPUT_DIR / "map_renewables_2024.csv", index=False)
print("✓ map_renewables_2024.csv  ->  Datawrapper choropleth map")
print(map_df.to_string(index=False))

In [ ]:
# CSV 2: Full time series, all EU-27
# Wide format: rows = years (2004 to 2024), columns = country names.
# Used as the reference dataset and for the all-countries line chart.
ts_wide = df_eu.pivot(index="year", columns="country", values="pct").round(2)
ts_wide.index.name = "year"
ts_wide.to_csv(OUTPUT_DIR / "timeseries_all_countries.csv")
print("✓ timeseries_all_countries.csv  ->  reference / all-countries chart")
print(f"  {ts_wide.shape[0]} years × {ts_wide.shape[1]} countries")

In [ ]:
# CSV 3: Leaders vs laggards + EU average
# 6 selected countries chosen to illustrate the divide:
#   Leaders:  Sweden, Denmark, Finland (consistently at top)
#   Laggards: Luxembourg, Belgium, Ireland (consistently at bottom)
# EU27 aggregate added as a benchmark reference line.
leaders  = ["Sweden", "Denmark", "Finland"]
laggards = ["Luxembourg", "Belgium", "Ireland"]

ts_divide = ts_wide[leaders + laggards].copy()

df_eu27["pct"] = pd.to_numeric(df_eu27["pct"], errors="coerce")
ts_divide["EU average"] = df_eu27.set_index("year")["pct"]

ts_divide.to_csv(OUTPUT_DIR / "timeseries_divide.csv")
print("✓ timeseries_divide.csv  ->  leaders vs laggards line chart")
print(ts_divide.tail(5))

In [ ]:
# CSV 4: Gap over time
# One row per year. Four columns:
#   max_pct: value of the top-ranked country that year
#   min_pct: value of the bottom-ranked country that year
#   avg_pct: EU-27 unweighted mean
#   gap_pp:  spread between max and min (the headline metric)
# Used for the area chart showing the widening divide.
# Note: max/min identify different countries in different years.
# Sweden leads throughout. Malta was the laggard until around 2015,
# then Belgium took the bottom position.
gap_df = pd.DataFrame({
    "year":    df_eu.groupby("year")["pct"].max().index,
    "max_pct": df_eu.groupby("year")["pct"].max().values.round(2),
    "min_pct": df_eu.groupby("year")["pct"].min().values.round(2),
    "avg_pct": df_eu.groupby("year")["pct"].mean().values.round(2),
    "gap_pp":  (df_eu.groupby("year")["pct"].max() -
                df_eu.groupby("year")["pct"].min()).values.round(2)
})
gap_df.to_csv(OUTPUT_DIR / "gap_over_time.csv", index=False)
print("✓ gap_over_time.csv  ->  area chart (widening gap)")
print(gap_df.to_string(index=False))

In [ ]:
# CSV 5: All 27 countries + EU average (for highlighted line chart)
# Same as CSV 2 but with EU average appended as an extra column.
# In Datawrapper: all 27 country lines set to grey,
# then Sweden, Finland, Belgium, Luxembourg individually recoloured.
# This creates the highlight effect used in the main line chart.
ts_all = ts_wide.copy()
ts_all["EU average"] = df_eu27.set_index("year")["pct"]
ts_all.to_csv(OUTPUT_DIR / "timeseries_all_highlighted.csv")
print("✓ timeseries_all_highlighted.csv  ->  highlighted line chart (all 27)")
print(f"  {ts_all.shape[1]} columns (27 countries + EU average) × {ts_all.shape[0]} years")

In [ ]:
# Summary
print("=== Output files ===")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    kb = f.stat().st_size / 1024
    print(f"  {f.name:<42} {kb:.1f} KB")

print()
print("Next step: upload each CSV to Datawrapper")
print("  map_renewables_2024.csv          ->  New Map -> Choropleth")
print("  timeseries_all_highlighted.csv   ->  New Chart -> Lines")
print("  gap_over_time.csv                ->  New Chart -> Lines + Fill areas")